<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-11-self-hosting/lesson-11.2-custom-fastapi/notebooks/GCP_Capstone_11.2_CustomFastAPI.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11.2 Custom FastAPI + vLLM — The Server Is the Kit's
**Netsetos GenAI Engineering — GCP Capstone** · Module 11 · rebuilt on the live lane, 10 September 2026

The production inference server - a FastAPI lifespan that loads the engine once, OpenAI-compatible schemas, SSE streaming, per-tenant keys and tiered rate limits, Presidio redaction, guided JSON - is seven files under `deploy/services/gemma-vllm`, and this notebook reads them from the clone instead of printing them from heredocs. It asserts the clone is clean and the backslashes are single (the kit's copies doubled them until 10 September, and the frames and regexes were silently broken), explains the two doors a request needs, runs the SSE generator against a fake engine and asserts the frames, classifies the lane's own text through whichever door exists, and puts the server's log row beside the lane's usage row.


## Setup
The kit, the roster member, Presidio's model, and the server's directory on the path.


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 google-cloud-bigquery==3.45.0 fastapi==0.141.1 pydantic==2.13.5 slowapi==0.1.10 presidio-analyzer==2.2.364 presidio-anonymizer==2.2.364 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "feat/lesson-4.8-live-evals"        # the demo branch; main is behind it

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
!python -m spacy download en_core_web_lg -q      # Presidio's NER model - the one the kit's Dockerfiles install
SERVER   = f"{KIT}/deploy/services/gemma-vllm"   # 11.2's seven files, read from the clone, never pasted
sys.path.insert(0, SERVER)
VLLM_URL = f"https://documind-vllm-{NUMBER}.{REGION}.run.app"   # make deploy-vllm (decision D3: optional)

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120) -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) as documind-ui-sa. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    r = requests.post(f"{url}{path}", json=body, headers={"Authorization": f"Bearer {documind_tools._id_token(url)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers):
    the body's `model` names what answered (a fallback included), the headers carry the cost the gateway priced."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def slm(path: str, body: dict | None = None, method: str = "POST", timeout: int = 240) -> tuple[int, dict | str, float]:
    """One call to the SLM's own doors (Ollama's /api/*, or its OpenAI-compatible /v1/*), timed - the first call after
    idle is the cold start."""
    t0 = time.time()
    r = requests.request(method, f"{SLM_URL}{path}", json=body, timeout=timeout,
                         headers={"Authorization": f"Bearer {documind_tools._id_token(SLM_URL)}"})
    try:
        return r.status_code, r.json(), time.time() - t0
    except ValueError:
        return r.status_code, r.text[:400], time.time() - t0

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="documind-api" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # the kit's builders and judges: run_eval, judge
sys.path.insert(0, f"{KIT}/deploy/services/slm")        # compare_backends, make_modelfile
print("helpers: api(), gateway(), slm(), service(), usage_rows(); the kit's evals/ and services/slm/ on sys.path")


## Cell 2: The server is the kit's
Seven files, a clean clone, and the doubled-backslash story asserted.


In [ ]:
# THE SERVER IS THE KIT'S. Seven files under deploy/services/gemma-vllm, read from the clone. This notebook used to
# print them from heredocs, and the kit carried copies typed by hand - three of which doubled every backslash (gap
# G2): the SSE generator ended each frame with a literal backslash-n, and the regex safety net could match nothing.
# Fixed at the source on 10 September 2026, and the two facts are asserted here rather than described.
files = sorted(f for f in os.listdir(SERVER) if not f.startswith("__"))
for f in files:
    print(f"  {f:22} {os.path.getsize(os.path.join(SERVER, f)):>6} bytes")
dirty = subprocess.run(["git", "-C", KIT, "status", "--porcelain", "--", "deploy/services/gemma-vllm"], capture_output=True, text=True).stdout
assert not dirty.strip(), f"the clone's copy is edited: {dirty}"
DOUBLED = chr(92) * 2                                   # a doubled backslash in the FILE - the shape G2 had
for f in ("streaming.py", "logging_module.py", "documind.py"):
    src = open(os.path.join(SERVER, f), encoding="utf-8").read()
    assert DOUBLED + "n" not in src and DOUBLED + "b" not in src, f"{f} carries doubled escapes again"
stream_src = open(os.path.join(SERVER, "streaming.py"), encoding="utf-8").read()
assert chr(92) + "n" + chr(92) + "n" in stream_src, "an SSE frame must end in two newlines"
print()
print(chr(10).join(l for l in stream_src.splitlines() if "data: " in l))
print()
print("seven files, a clean clone, single backslashes: the frames end in two real newlines and the regexes can match")


### The lifespan, the door and the shapes, from the files
The lines that carry each, printed from the clone; then the image: vLLM's base, the pins on top, one worker.


In [ ]:
# THE LIFESPAN, THE DOOR AND THE SHAPES, FROM THE FILES. main.py loads the engine ONCE in a FastAPI lifespan and puts the
# limiter on app.state before the middleware, in that order; auth.py is door 2 - the X-API-Key hashed, looked up in
# api_keys/, resolved to a tenant's tier, turned into a rate limit by a function slowapi calls by the NAME of its
# parameter; schemas.py is OpenAI's request and response shape in Pydantic v2, which is why every client in Module 11
# can stand in front of it. The lines that carry each, printed from the clone rather than retyped.
def lines(rel, *needles):
    src = open(f"{KIT}/deploy/services/{rel}", encoding="utf-8").read().splitlines()
    print(f"# {rel}  ({len(src)} lines)")
    print(chr(10).join(l for l in src if any(n in l for n in needles)))
    print()
lines("gemma-vllm/main.py", "async def lifespan", "AsyncLLMEngine.from_engine_args", "app = FastAPI", "app.state.limiter", "add_middleware", "@app.", "@limiter.limit")
lines("gemma-vllm/auth.py", "APIKeyHeader(", "sha256", 'collection("api_keys")', "TIER_LIMITS", "Limiter(", "def tier_rate_limit", "def get_tenant")
lines("gemma-vllm/schemas.py", "class ")


In [ ]:
# THE IMAGE. The Dockerfile starts from vLLM's own image - CUDA, torch and vllm pinned by the base, not by pip - installs
# the server's pins on top, resets the base's ENTRYPOINT so uvicorn is the command, and runs ONE worker, because a
# GPU's memory is not shareable across processes. validate.py parses and lints this file and cannot build it (the base
# is gigabytes and needs a GPU to run), which is why make build-vllm sends it to Cloud Build instead.
for rel in ("services/gemma-vllm/Dockerfile", "services/gemma-vllm/requirements.txt"):
    text = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read()
    print(f"# {rel}")
    print(chr(10).join(l for l in text.splitlines() if l.strip() and not l.startswith("#")))
    print()


## Cell 3: Two doors
Cloud Run IAM at the front (the lane's identity), the API key inside (a product's), and the slowapi signature rule checked.


In [ ]:
import inspect
import auth, schemas

# TWO DOORS, AND WHY A REQUEST NEEDS BOTH. The lane's identity is Cloud Run IAM: the caller presents a Google ID token
# for the service's URL and the platform refuses everything else before a byte reaches the server (the deploy says
# --no-allow-unauthenticated; the gateway's account and the UI's are invited). The server's identity is a PRODUCT's:
# an X-API-Key hashed into Firestore api_keys, a tenant with a tier, a rate limit per tier. The first says "this
# workload may call us"; the second says "which customer is this, and how fast". An endpoint sold to more than one
# customer needs both, and a learner who tries one without the other gets a 403 from the platform or a 401 from the app.
r = requests.get(f"{VLLM_URL}/health", timeout=15)
print("door 1, IAM    :", f"HTTP {r.status_code} without a token" + (" - refused at the platform" if r.status_code in (401, 403) else " - the engine is not deployed (decision D3)"))
print("door 2, product: X-API-Key ->", auth.hash_api_key("sk-demo-key")[:16] + "... -> api_keys/{hash} -> tenants/{tenant_id} -> tier")
for tier, limit in auth.TIER_LIMITS.items():
    print(f"  {tier:11} {limit}")
assert auth.tier_rate_limit("anonymous") == "10/minute"
# slowapi calls the limit function by the NAME of its parameter: `key` receives key_func(request); any other name is
# called with nothing and raises at request time, in production. The file's docstring records it; this line checks it.
assert list(inspect.signature(auth.tier_rate_limit).parameters) == ["key"]
print()
print("the request schema is OpenAI's:", list(schemas.ChatCompletionRequest.model_fields), "- 11.1's client, 11.3's gateway and 11.4's SLM all speak it")


## Cell 4: Streaming, exercised
The kit's generator against a fake engine: the frames end in two newlines and the suffixes rebuild the text.


In [ ]:
from types import SimpleNamespace
from streaming import generate_sse_stream

# STREAMING, EXERCISED. vLLM's RequestOutput carries the CUMULATIVE text; an SSE client wants the suffix; a frame is
# `data: <json>` followed by two newlines; the stream ends with `data: [DONE]`. A fake engine plays three cumulative
# outputs into the kit's real generator, and the assertions are on the bytes a browser would parse - the cell that
# went red on the doubled backslashes and green after the fix.
class FakeEngine:
    async def generate(self, prompt, sampling, request_id):
        for text, fin in (("The notice", None), ("The notice period", None), ("The notice period is 60 days.", "stop")):
            yield SimpleNamespace(outputs=[SimpleNamespace(index=0, text=text, finish_reason=fin)])
    async def abort(self, request_id):
        print("aborted", request_id)

class FakeRequest:
    async def is_disconnected(self):
        return False

frames = []
async for f in generate_sse_stream(FakeEngine(), "req-1", "documind-inference", "prompt", None, FakeRequest()):
    frames.append(f)
    print(repr(f))
assert all(f.endswith(chr(10) * 2) for f in frames), "a frame ends in two newlines, not a literal backslash-n"
deltas = [json.loads(f[6:])["choices"][0]["delta"].get("content", "") for f in frames[:-1]]
assert "".join(deltas) == "The notice period is 60 days.", deltas
assert json.loads(frames[-2][6:])["choices"][0]["finish_reason"] == "stop" and frames[-1].startswith("data: [DONE]")
print(f"\n{len(frames)} frames: a role frame, three suffix deltas, a finish frame, [DONE] - and the suffixes rebuild the text")


## Cell 5: Guided JSON on the lane's own text


In [ ]:
import ast
from typing import Literal
from pydantic import BaseModel, ValidationError

# GUIDED JSON, AND WHAT "GUARANTEED" MEANS. /v1/documind/classify hands vLLM a JSON Schema as StructuredOutputsParams:
# the engine constrains every token, so the reply IS the schema and there is no retry-on-parse loop. documind.py
# imports vllm, so the result model is re-declared here and asserted identical to the file's; the text is the
# lane's own - one chunk from the one retrieve(); the door is the engine's when it is deployed, and otherwise the
# gateway's documind-inference route, where Ollama's json mode is the model's promise rather than the decoder's.
# Three tries validated by the same model class: that count is the difference between the two.
class ClassificationResult(BaseModel):
    category: Literal["invoice", "contract", "report", "letter", "other"]
    confidence: float
    reasoning: str

tree = ast.parse(open(os.path.join(SERVER, "documind.py"), encoding="utf-8").read())
fields = [n.target.id for c in tree.body if isinstance(c, ast.ClassDef) and c.name == "ClassificationResult"
          for n in c.body if isinstance(n, ast.AnnAssign)]
assert fields == list(ClassificationResult.model_fields), fields

got = documind_tools.retrieve("What is the notice period for a confirmed E3?", tenant_id=TENANT, top_k=1, brain="direct")
text = (got.get("citations") or [{}])[0].get("quote") or "No document was retrieved."
prompt = f"Classify this document into: invoice, contract, report, letter, other.\n\nDocument:\n{text[:4000]}\n\nRespond with JSON:"
print("the chunk:", text[:110].replace(chr(10), " "), "...")
print()
valid, status = 0, 0
for i in range(3):
    status, body, headers = gateway("documind-inference", prompt, json_mode=True, max_tokens=200)
    raw = body["choices"][0]["message"]["content"] if status == 200 else str(body)
    try:
        res = ClassificationResult.model_validate_json(raw)
        valid += 1
        print(f"  try {i + 1}: {res.category:9} {res.confidence:.2f}  {res.reasoning[:56]!r}  ({body.get('model')})")
    except (ValidationError, ValueError):
        print(f"  try {i + 1}: not the schema -> {raw[:80]!r}")
assert status == 200, f"the route did not answer: HTTP {status} {raw[:120]}"
print(f"\n{valid}/3 valid. Behind vLLM's structured outputs that is 3/3 by construction; behind json mode it is the model's habit -")
print("and the API's own path treats every reply the same way: ModelDraft validated, a bad one a 502, never a retry loop that hides the rate.")


## Cell 6: The log row, redacted - and the row the lane wants


In [ ]:
import re
import logging_module

# THE LOG ROW, REDACTED - AND THE ROW THE LANE WANTS. The server logs one row per request after the response is sent
# (a BackgroundTask): the prompt redacted by Presidio, with a regex safety net behind it. Presidio's engines load
# here (the spaCy model); the safety net's three patterns are pulled from the source and run on their own - G2's
# gate from the other side; and the row's shape is put beside the API's usage row, which tenant_daily reads:
# cost_usd, latency_ms, model_backend and brain, none of which this row carries yet.
sample = "Email me at priya@acme.example, PAN ABCPE1234F, phone 987-654-3210, about the notice period."
out = logging_module.redact_pii(sample)
print("redacted:", out)
assert "priya@acme.example" not in out, out
print("the PAN was", "caught" if "ABCPE1234F" not in out else "NOT caught by Presidio's IN_PAN - the DLP sampler in 11.3 exists for exactly this gap")
src = open(logging_module.__file__, encoding="utf-8").read()
patterns = re.findall('re[.]sub[(]r"([^"]+)"', src)
assert len(patterns) == 3, patterns
for pat, probe in zip(patterns, ("mail priya@acme.example now", "call 987-654-3210", "SSN 123-45-6789")):
    assert re.search(pat, probe), f"the safety net does not match its own case: {pat!r}"
print(f"the safety net's {len(patterns)} patterns match an email, a phone and an SSN on their own (they matched nothing before 10 September)")

server_row = {"request_id": "req-1", "tenant_id": TENANT, "model": "google/gemma-3-4b-it", "prompt_tokens": 812, "completion_tokens": 44, "prompt_redacted": out[:60]}
lane_rows = usage_rows(minutes=60 * 24, limit=1)
lane_keys = set(lane_rows[0]) if lane_rows else {"tenant", "model", "model_backend", "cost_usd", "latency_ms", "tokens_in", "tokens_out", "brain", "surface"}
print()
print("the server's row :", sorted(server_row))
print("the lane's row   :", sorted(lane_keys))
print("missing for tenant_daily:", sorted(k for k in ("cost_usd", "latency_ms", "model_backend", "brain") if k not in server_row),
      "- which is why, on the lane, the API logs the row and the engine stays a backend")


## Where this goes
- **11.3** puts a route in front of this door - and the PII decision moves to the gateway, made once, where every route passes.
- **11.4** deploys 10.5's model behind the same OpenAI-compatible shape and prices it; the API's usage row, not this server's log row, is what tenant_daily reads.

## ✅ Lesson 11.2 complete
- ✅ Seven files read from the clone; the clone clean; single backslashes asserted (G2)
- ✅ main.py, auth.py and schemas.py excerpted from the files; the Dockerfile and the pins printed
- ✅ Two doors: IAM at the platform, the hashed key and the tier inside; the slowapi `key` rule checked
- ✅ The SSE generator run against a fake engine: frames, suffixes, finish, [DONE]
- ✅ Guided JSON on a chunk from the one retrieve(), validated by the file's own schema
- ✅ Redaction through Presidio with the safety net proven; the row shape beside the lane's
